In [ ]:
# tool create
from langchain_core.tools import tool
import requests
from langchain_core.messages import HumanMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated

# LLM ko sirf woh arguments expose karo jo usse provide karne hain; baaki arguments application/runtime inject kare.
# Ek aur important point
# InjectedToolArg ka matlab automatically previous tool ka output next tool mein inject kar dena nahi hai.
from langchain_core.tools import InjectedToolArg


@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
        This fucntion fetches the currency conversion factor between a given base currency and a target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/{}/pair/{base_currency}/{target_currency}'
    response = requests.get(url)
    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) ->float:
    """ given the currency conversion rate this function calculates the target currency value from a given base currency value"""
    return base_currency_value * conversion_rate





In [2]:
get_conversion_factor.invoke({
    "base_currency": "USD",
    "target_currency": "INR"
})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1791417601,
 'time_last_update_utc': 'Thu, 08 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1791504001,
 'time_next_update_utc': 'Fri, 09 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.8339}

In [3]:
convert.invoke({
    "base_currency_value": 10,
    "conversion_rate": 85.16
})

851.5999999999999

In [4]:
# tool binding
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:4b")
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [5]:
messages = []
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr')]

In [6]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [7]:
ai_message = llm_with_tools.invoke(messages)

In [8]:
ai_message
messages.append(ai_message)
print(ai_message)

content='' additional_kwargs={} response_metadata={'model': 'qwen3:4b', 'created_at': '2026-10-08T05:37:28.69171016Z', 'done': True, 'done_reason': 'stop', 'total_duration': 596044747631, 'load_duration': 788263, 'prompt_eval_count': 235, 'prompt_eval_duration': 152301000, 'eval_count': 2321, 'eval_duration': 595888138000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'} id='lc_run--01a119fb-2825-7e61-a476-d59392a8794a-0' tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'b1dd33a7-2199-4ae6-bb1e-85e8ca4d5daa', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10}, 'id': '520c04d1-1648-4677-826a-689309ea304b', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 235, 'output_tokens': 2321, 'total_tokens': 2556}


In [9]:
import json
for tool_call in ai_message.tool_calls:
    print("tool_call",tool_call)
    # execute the 1st tool and get the value of conversion rate
    
    if tool_call['name'] == "get_conversion_factor":
        tool_message1 = get_conversion_factor.invoke(tool_call)
        # fetch this conversion rate
        conversion_rate = (json.loads(tool_message1.content)['conversion_rate'])
        # append this tool message to message list
        messages.append(tool_message1)
        print("tool_message1", tool_message1)
    # execute the second tool using the conversion rate from tool1
    if tool_call['name'] == "convert":
        # fetch teh currrent argument
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        messages.append(tool_message2)


tool_call {'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'b1dd33a7-2199-4ae6-bb1e-85e8ca4d5daa', 'type': 'tool_call'}
tool_message1 content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1791417601, "time_last_update_utc": "Thu, 08 Oct 2026 00:00:01 +0000", "time_next_update_unix": 1791504001, "time_next_update_utc": "Fri, 09 Oct 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 96.8339}' name='get_conversion_factor' tool_call_id='b1dd33a7-2199-4ae6-bb1e-85e8ca4d5daa'
tool_call {'name': 'convert', 'args': {'base_currency_value': 10}, 'id': '520c04d1-1648-4677-826a-689309ea304b', 'type': 'tool_call'}


In [10]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-10-08T05:37:28.69171016Z', 'done': True, 'done_reason': 'stop', 'total_duration': 596044747631, 'load_duration': 788263, 'prompt_eval_count': 235, 'prompt_eval_duration': 152301000, 'eval_count': 2321, 'eval_duration': 595888138000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a119fb-2825-7e61-a476-d59392a8794a-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'b1dd33a7-2199-4ae6-bb1e-85e8ca4d5daa', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10, 'conversion_rate': 96.8339}, 'id': '520c04d1-1648-4677-826a-689309ea304b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={

In [11]:
llm_with_tools.invoke(messages)

AIMessage(content='The currency conversion factor between USD and INR is **96.8339**. \n\nUsing this rate, **10 USD** converts to **968.34 INR** (rounded to two decimal places).', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-10-08T05:39:17.276591314Z', 'done': True, 'done_reason': 'stop', 'total_duration': 108220054993, 'load_duration': 1350957, 'prompt_eval_count': 498, 'prompt_eval_duration': 16275110000, 'eval_count': 454, 'eval_duration': 91661128000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a11a04-41df-7641-9a48-a53ad65564a1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 498, 'output_tokens': 454, 'total_tokens': 952})